# Guidelines for data treatment with SCARLET

This notebook, present how to treat data with SCARLET python API

## 1. Workflow initialization: 

In [ ]:
from scarlet.workflow.context import initialize_workflow_context_from_raw_directory, WorkflowContext

RAW_DIR = "/home/achennev/python/scarlet/data/SANSLLB/raw_june" # change this to the path where your raw data is located
OUTPUT_DIR = "/home/achennev/python/scarlet/data/SANSLLB/out_june_v3" # change this to the path where you want to store the output data
INSTRUMENT_NAME = "sansllb" # instrument name, used to find the correct configuration files

RAW_DIR, OUTPUT_DIR


Initialize the work flow context using the `initialize_workflow_context_from_raw_directory` function. This function perform several task : 

 - Read all the data files present in the raw data driectory, convert them into a nexus file with a generic file architecure and save them into the output_directory. 

 - Make a list of instrument configuration used during the experiment (collimation, detector distance, wavelength)

 - Detect if data file corresponds to "empty_cell", "dark", "water" or "empty_beam" measurements based on the sample name.


In [ ]:
w = initialize_workflow_context_from_raw_directory(
    RAW_DIR,
    instrument_name=INSTRUMENT_NAME,
    transmission_strategy="opaque_beamstop",  #Can be "semi_transparent_beamtop" to compute transmission with scattering files
    output_dir=OUTPUT_DIR,
    overwrite=True,
)
# If orverwrite is set to True, the output directory will be cleared before running the workflow. If set to False, the workflow will not run if the output directory already exists.

## 2. Save workflow and filter files
Sometimes, the raw_data folder contains many files that are not necessary for the data treatement and complexify the workflow operation. The 'WorkflowContext ' shall only contains runs intended to be treated and references (empty_cell,...).

For this prupose, a manual filtering or the edition of a csv file is possible.

First save the current state of the workflow in a csv file using the command:

In [ ]:
w.write_runs_table_csv('runs_table.csv', overwrite=True)

After editing the csv file, you can run the workflow with the following code in order to load the filtered csv files. The runs can be displayed using the `w.runs_table()` command.

In [ ]:
RUN_TABLE_CSV = 'runs_filtered.csv'
w.update_from_runs_table_csv(RUN_TABLE_CSV)
w.runs_table()


The entity column serves as a sample description. It can be set to :
- sample : normal sample
- water: water file allowing to compute flatfield
- empty_cell : empty_cell for substraction
- dark: B4C, Cd file for substraction
- empty_beam : note that a transmission empty_beam is required to compute the transmissions. 

The mode column can also be changed in case of miss recognition. 

The config_id column is just a simple identification string per confiruration used. The configurations parameters can be displayed using the `WorkflowContext.configurations_table()` method

In [ ]:
w.configurations_table()

The `WorkflowContext` can be saved a a nexus file for futur use using the following commands:

In [ ]:

w.save("workflow.nxs", overwrite=True)
# load the workflow context from the saved file
w = WorkflowContext.load("workflow.nxs")

## 3. Transmission computation

The computation of the detector is performed using the detector named `detector0` in the nexus files refering to the central detector. During the initialization of the `WorkflowContext`, the ROI for transmission measurements is automatically detected.

The ROIs per config can be retrieved and modified using the following methods:

In [ ]:
# read roi
print(w.get_roi('config_9'))
# set roi
# w.set_roi('config_10', [61, 65, 62, 64])  # Example ROI coordinates
# w.set_roi('config_9', [60, 64, 61, 63])  # Example ROI coordinates

The computation of the transmission is perfomd using the `WorkflowContext.compute_transmissions()` method:

In [ ]:
w.compute_transmissions()
# save transmission in csv file once computed
w.write_runs_table_csv("runs_bs_semi_2.csv",overwrite=True)
w.runs_table()

In some experiments the transmission is measured only once for several scattering configurations, typically when the wavelength, attenuator setup, and transmission geometry are equivalent. In that case, compute the available transmissions first, then declare which configuration should provide the transmission values for another configuration.

Use `WorkflowContext.set_transmission_source(config_id=target_config, source_config_id=source_config)`. The first argument is the configuration that needs a transmission value; the second argument is the configuration where the transmission was measured. Once declared, `get_transmission(sample_name, target_config)` and the reduction pipeline will automatically use the transmission stored on `source_config`. The same mapping is also used for empty-cell transmissions and for water-flatfield preparation.




The mapping is saved in the workflow context when calling `w.save(...)`, so it is restored when the workflow is loaded again.


In [ ]:
w.compute_transmissions()

# Reuse transmissions measured in config_4 for samples reduced in config_2 and config_3.
w.set_transmission_source(config_id="config_2", source_config_id="config_4")
w.set_transmission_source(config_id="config_3", source_config_id="config_4")

# Optional checks.
w.get_transmission_source("config_2")
w.resolve_transmission_config("config_2")
w.get_transmission(sample_name="acb75 75D", config_id="config_2")

## 4. Masking


The edition of masks can be done using the `scarlet viewer` GUI. To run this graphical interface just run `scarlet viewer` command in a terminal with the appropriate virtual environnment activated. The mask files shall be saved in the output directory in order to be found by the workflow.

Each mask file stores the configuration over which they have been drawn. The `WorkflowContext` can automatically adress them to a specc

In [ ]:
w.attach_mask_bundles_from_output_dir()
# display the mask files for a specific configuration
w.mask_files

Manual attibution of a mask file can be done using the following command:

In [ ]:
# w.set_mask_file(config_id='config_10', file_path='/path/to/mask_file')

## 5. Flatfield / water normalization

The `water` entity in the run table is used to generate one flatfield per configuration. Current flatfields also contain the attenuation-corrected direct flux and one `normalisation_factor` per detector. Masked, non-finite, and non-positive pixels are excluded from the detector mean used for that factor.

Converted files may contain an optional `NXattenuator` with `selection` and `attenuator_transmission`. The transmission is `I_attenuated / I_incident` and must be in `(0, 1]`; old files without this group are treated as unattenuated.

Build the flatfield for each configuration that supplies one:

In [ ]:
w.build_water_flatfield(config_id="config_10")
w.build_water_flatfield(config_id="config_9")

The flatfield is saved in the output directory. It can also be assigned manually:

In [ ]:
# set flatfield file for a specific configuration
# w.set_flatfield(config_id="config_10", file_path="flat_field_path")
# get the flatfield file for a specific configuration
w.get_flatfield(config_id="config_9")

If no water file is available at a given configuration we can declare to use the flatfield of one config to another one using the following command.

In [ ]:
w.set_flatfield_source(config_id="config_9", source_config_id="config_10")
w.get_flatfield(config_id="config_9")

## 6. Reduction pipeline

Once the workflow is set, the data reduction can be performed using a `Pipeline`. The default pipeline performs the following steps:
- subtract dark and empty-cell references;
- divide by the water flatfield and normalize by solid angle;
- normalize by sample thickness;
- convert to absolute intensity in `cm^-1` using the detector prefactor and direct-flux ratio;
- integrate azimuthally with detector-specific bin counts and linear or logarithmic q spacing;
- save processed detectors in the NeXus `/processed` entry;
- save one text curve per configuration and detector.

Reduce one sample and configuration with explicit integration settings:

In [ ]:
from scarlet.workflow.pipeline import ReductionPipeline
pipeline = ReductionPipeline.default(
    w,
    n_bins={0: 120, 1: 30, 2: 30},
    q_scale="linear",
)
state = pipeline.run_for_sample(sample_name="ludox_SM30", config_id="config_9")

Or reduce every registered scattering sample:

In [ ]:
pipeline.run_all()

In [ ]:
# w.save("workflow.nxs")

w.load("workflow.nxs")

`n_bins` may also be one integer shared by every detector. Set `q_scale="log"` for logarithmic q-bin boundaries. Flatfields created before absolute-calibration metadata was introduced must be rebuilt.

## 7. Stitching configurations

`scale_on` selects the processed reference configuration. `normalization_factor` multiplies the final `I` and `I_error` values without changing q or `q_error`.

In [ ]:
from scarlet.workflow.pipeline import StichingPipeline
stitch = StichingPipeline(w)
stitch.run_new(
    scale_on="config_10",
    normalization_factor=1.0,
)